# Transfer Learning for SteelBlast Image Classification

Using pre-trained neural networks (VGG16, InceptionV3, ResNet50) to classify steel surfaces as "good" or "not-good" for shot-blasting quality assessment.

## 1. Import Required Libraries

In [ ]:
from __future__ import annotations

import os
import json
import sys
import tempfile
from pathlib import Path
from dataclasses import dataclass

# Set matplotlib backend before importing
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

# Reproducibility seed
RANDOM_SEED = 42
os.environ.setdefault('PYTHONHASHSEED', str(RANDOM_SEED))
np.random.seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import VGG16, InceptionV3, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau




## Pipeline Configuration

This cell initializes runtime and pipeline settings before dataset loading and model training.

#### Approach
- Detect whether execution is in Colab (`detected_in_colab`) to support portable behavior.
- Define the main pipeline configuration through `PipelineConfig`.
- Apply local-only environment settings when `local_env=True`.
- Detect available GPUs and enforce the user policy from `enable_gpu`.
- Optionally enable mixed precision when GPU execution is active.
- Print the resolved runtime state for traceability.

#### Parameters
##### `PipelineConfig`
- `dataset_dir: Path`
  - Root folder containing train/test class subfolders.
- `model_choice: str`
  - Backbone selector: `VGG16`, `InceptionV3`, or `ResNet50`.
- `in_colab: bool`
  - Indicates whether code is running in Google Colab.
- `local_env: bool`
  - Enables local-only environment settings (for example Apple Silicon tuning).
- `enable_gpu: bool`
  - User-controlled flag to allow GPU execution.
- `use_gpu: bool`
  - Effective GPU status after availability checks and policy application.


In [ ]:
detected_in_colab = "google.colab" in sys.modules
print(f"Running in Colab: {detected_in_colab}")
print(f"TensorFlow version: {tf.__version__}")



@dataclass(frozen=True)
class PipelineConfig:
    """Dataset path and runtime environment flags."""
    dataset_dir: Path = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
    model_choice: str = 'ResNet50' # 'VGG16', 'InceptionV3', or 'ResNet50'
    in_colab: bool = False
    local_env: bool = True # Set True to apply Apple-Silicon local env vars
    enable_gpu: bool = False # User-controlled: set True to request GPU use
    use_gpu: bool = False # Effective value resolved at runtime




LABELS = {"good": 0, "not-good": 1}
CLASS_NAMES = ["good", "not-good"]

pipeline_config = PipelineConfig(in_colab=detected_in_colab, local_env=not detected_in_colab)



# Execute local-only environment variables only when local_env is enabled.
if pipeline_config.local_env:
    os.environ.setdefault("TF_GPU_ALLOCATOR", "metal")
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    print("Applied local environment variables (TF_GPU_ALLOCATOR, TF_CPP_MIN_LOG_LEVEL)")
else:
    print("Skipped local environment variables because local_env=False")

# Detect and optionally enable GPU based on config flag
detected_gpus = tf.config.list_physical_devices('GPU')
gpu_available = len(detected_gpus) > 0

if gpu_available and not pipeline_config.enable_gpu:
    # Disable GPU visibility so TensorFlow runs on CPU when the flag is off.
    try:
        tf.config.set_visible_devices([], 'GPU')
        detected_gpus = []
        gpu_available = False
        print("GPU disabled by AppConfig.enable_gpu=False")
    except RuntimeError as e:
        print("Could not disable GPU after initialization:", e)

use_gpu = pipeline_config.enable_gpu and gpu_available

if use_gpu:
    try:
        for gpu in detected_gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf.config.optimizer.set_jit(False)
        print("Enabled memory growth for GPU; XLA JIT disabled")
    except RuntimeError as e:
        print("Could not set GPU memory growth:", e)

if use_gpu:
    policy = tf.keras.mixed_precision.Policy("mixed_float16")
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f"Enabled mixed precision policy: {policy.name}")
else:
    print("Mixed precision disabled because GPU use is off or no GPU was found")


print("JIT enabled after config:", tf.config.optimizer.get_jit())
print(f"Model: {pipeline_config.model_choice}")
print(f"Dataset directory: {pipeline_config.dataset_dir}")
print(f"In Colab: {pipeline_config.in_colab}")
print(f"Local env flag: {pipeline_config.local_env}")
print(f"GPU available: {gpu_available}")
print(f"GPU enabled by flag: {pipeline_config.enable_gpu}")
print(f"Using GPU: {pipeline_config.use_gpu}")


## Load and Pre-process Dataset

This cell defines preprocessing settings, loads dataset splits from disk, applies model-specific preprocessing, and builds efficient `tf.data` pipelines for training, validation, and testing.

#### Approach
- Define image and split settings with `PreprocessConfig`.
- Adapt image size to the selected backbone (`pipeline_config.model_choice`).
- Load image paths and class labels from the dataset folder structure.
- Create a stratified train/validation split to preserve class balance.
- Select the correct model-specific `preprocess_input` function.
- Decode, resize, preprocess, augment, batch, cache, and prefetch data using `tf.data`.

#### Parameters
##### `PreprocessConfig`
- `image_size: tuple`
  - Input resolution for resizing images before preprocessing.
  - Uses `(224, 224)` for `ResNet50`/`VGG16` and `(299, 299)` for `InceptionV3`.
- `batch_size: int`
  - Number of samples per batch.
- `validation_split: float`
  - Fraction of training data reserved for validation.



In [ ]:
@dataclass(frozen=True)
class PreprocessConfig:
    """Image size, batching, and train/val split parameters."""
    image_size: tuple = (224, 224) # VGG16/ResNet50: 224x224 | InceptionV3: 299x299
    batch_size: int = 32
    validation_split: float = 0.2

preprocess_config = PreprocessConfig(batch_size=32, validation_split=0.2)

# Adjust image size based on model choice
if pipeline_config.model_choice == 'InceptionV3':
    preprocess_config.image_size=(299, 299)

print(f"Image size for model {pipeline_config.model_choice}: {preprocess_config.image_size}")
print(f"Batch size: {preprocess_config.batch_size}")
print(f"Validation split: {preprocess_config.validation_split}")

def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    """Load image paths and labels for train or test split."""
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)

# Load dataset splits
train_paths, y_train = load_split(pipeline_config.dataset_dir, "train")
test_paths, y_test = load_split(pipeline_config.dataset_dir, "test")

print(f"Train set: {len(train_paths)} images")
print(f"  - Good: {(y_train == 0).sum()}")
print(f"  - Not-good: {(y_train == 1).sum()}")
print(f"\nTest set: {len(test_paths)} images")
print(f"  - Good: {(y_test == 0).sum()}")
print(f"  - Not-good: {(y_test == 1).sum()}")

# Split train into train and validation

train_paths, val_paths, y_train, y_val = train_test_split(
    train_paths, y_train, test_size=preprocess_config.validation_split, 
    random_state=42, stratify=y_train
)

print(f"\nAfter train/val split:")
print(f"Train: {len(train_paths)}, Validation: {len(val_paths)}, Test: {len(test_paths)}")

# Preprocessing function selection based on model choice
def get_preprocess_fn(model_choice: str):
    if model_choice == 'VGG16':
        return vgg16_preprocess
    if model_choice == 'InceptionV3':
        return inception_preprocess
    return resnet_preprocess

preprocess_fn = get_preprocess_fn(pipeline_config.model_choice)

print(f"Preparing tf.data datasets with {pipeline_config.model_choice} preprocessing...")

# resize and preprocess images
def decode_and_preprocess(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, preprocess_config.image_size)
    image = tf.cast(image, tf.float32)
    image = preprocess_fn(image)
    return image, label

# Data augmentation function
# TODO validate robustness over lighting conditions
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.10)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return image, label


def build_dataset(image_paths, labels, batch_size, is_training=False):
    image_paths = [str(path) for path in image_paths]
    labels = labels.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(decode_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if is_training:
        dataset = dataset.shuffle(buffer_size=min(len(image_paths), 1000), reshuffle_each_iteration=True)
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.cache()
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = build_dataset(train_paths, y_train, preprocess_config.batch_size, is_training=True)
val_dataset = build_dataset(val_paths, y_val, preprocess_config.batch_size)
test_dataset = build_dataset(test_paths, y_test, preprocess_config.batch_size)

print(f"Train dataset batches: {len(train_paths) // preprocess_config.batch_size} (+ remainder)")
print(f"Validation samples: {len(val_paths)}")
print(f"Test samples: {len(test_paths)}")


## Transfer Learning
 ### TODO 
- explain restnet outperformed other models
- refer to Fu J, Zhu X, Li Y (2019) for approach

This cell defines transfer-learning settings and loads the selected pretrained backbone (`VGG16`, `InceptionV3`, or `ResNet50`) with configurable freezing behavior.

#### Approach
- Define backbone-loading options in `TransferLearningConfig`.
- Build the selected backbone using `pipeline_config.model_choice`.
- Set `include_top=False` to remove the original ImageNet classifier head.
- Load pretrained weights from `imagenet` (or another configured source).
- Apply `trainable` to freeze or unfreeze backbone layers for feature extraction or fine-tuning.
- Print resolved settings and model statistics for traceability.

#### Parameters
##### `TransferLearningConfig`
- `include_top: bool`
  - Whether to keep the original classifier head from the pretrained model.
  - Typically `False` when adding a custom task-specific head.
- `weights: str`
  - Weight initialization source (for example `imagenet` or `None`).
- `trainable: bool`
  - `False` freezes the backbone (feature extraction mode).
  - `True` unfreezes the backbone (fine-tuning mode).


In [ ]:
@dataclass(frozen=True)
class TransferLearningConfig:
    """Configuration for loading and freezing the pretrained backbone."""
    include_top: bool = False
    weights: str = 'imagenet'
    trainable: bool = False
    
transfer_learning_config = TransferLearningConfig()
print(f"Transfer learning include_top: {transfer_learning_config.include_top}")
print(f"Transfer learning weights: {transfer_learning_config.weights}")
print(f"Transfer learning trainable: {transfer_learning_config.trainable}")

# Load pre-trained model with configured transfer-learning parameters
print(f"Loading {pipeline_config.model_choice} with {transfer_learning_config.weights} weights...")

if pipeline_config.model_choice == 'VGG16':
    base_model = VGG16(
        input_shape=(*preprocess_config.image_size, 3),
        include_top=transfer_learning_config.include_top,
        weights=transfer_learning_config.weights
    )
elif pipeline_config.model_choice == 'InceptionV3':
    base_model = InceptionV3(
        input_shape=(*preprocess_config.image_size, 3),
        include_top=transfer_learning_config.include_top,
        weights=transfer_learning_config.weights
    )
else:  # ResNet50
    base_model = ResNet50(
        input_shape=(*preprocess_config.image_size, 3),
        include_top=transfer_learning_config.include_top,
        weights=transfer_learning_config.weights
    )

# Freeze or unfreeze base model layers based on transfer-learning config
base_model.trainable = transfer_learning_config.trainable
print(f"Base model loaded with {len(base_model.layers)} layers")
print(f"Base model trainable: {base_model.trainable}")
print(f"Trainable parameters in base model: {base_model.count_params():,}")


## Modelling

This cell builds the final binary classifier by attaching a custom dense head to the pretrained backbone, then compiles and trains it with callbacks for convergence control.

#### Approach
- Convert spatial feature maps into a compact vector using global pooling. Lin et al., 2013, "Network In Network"
- Learn task-specific nonlinear representations with dense layers.
- Apply dropout regularization to reduce overfitting.Srivastava et al., 2014
- Output class probability for binary classification (`not-good` as class 1).
- Select optimizer dynamically from configuration (`adam` or `sgd`).
- Compile with `binary_crossentropy` and `accuracy`.
- Train with validation monitoring, early stopping, checkpointing, and adaptive learning-rate reduction.

#### Parameters
##### `ModellingConfig`
- `epochs: int`
  - Maximum number of training epochs.
- `learning_rate: float`
  - Initial learning rate for the selected optimizer.
- `dense_activation: str`
  - Activation function used in hidden dense layers.
- `classification_activation: str`
  - Output-layer activation for binary prediction (typically `sigmoid`).
- `optimizer_function: str`
  - Optimizer name used at compile time (`adam` or `sgd`).
- `early_stopping_patience: int`
  - Number of epochs with no validation improvement before stopping.
- `reduce_lr_patience: int`
  - Number of plateau epochs before reducing learning rate.


In [ ]:
@dataclass(frozen=True)
class ModellingConfig:
    """Model architecture and optimization controls."""
    # Maximum number of epochs to train the model
    epochs: int = 20
    # Learning rate for the optimizer
    learning_rate: float = 0.001
    # Activation function for dense layers
    dense_activation: str = 'relu'
    # Activation function for the output layer in binary classification
    classification_activation: str = 'sigmoid'
    # Optimizer function to use
    optimizer_function: str = 'adam'
    # Patience for early stopping
    early_stopping_patience: int = 10
    # Patience for reducing learning rate on plateau
    reduce_lr_patience: int = 5


model_config = ModellingConfig()

# Build custom classification model on top of base model
print("Building classification model...")

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation=model_config.dense_activation, name='dense_1'),
    layers.Dropout(0.5),
    layers.Dense(128, activation=model_config.dense_activation, name='dense_2'),
    layers.Dropout(0.3),
    # Output layer for binary classification
    layers.Dense(1, activation=model_config.classification_activation, dtype='float32', name='output')
])

print(f"Model architecture:")
model.summary()

# Compile the model
if model_config.optimizer_function.lower() == 'adam':
    optimizer = keras.optimizers.Adam(learning_rate=model_config.learning_rate)
elif model_config.optimizer_function.lower() == 'sgd':
    optimizer = keras.optimizers.SGD(learning_rate=model_config.learning_rate, momentum=0.9)
else:
    raise ValueError(f"Unsupported optimizer_function: {model_config.optimizer_function}")

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully")
print(f"Optimizer: {model_config.optimizer_function.upper()} (lr={model_config.learning_rate})")
print("Loss: binary_crossentropy")
print("Metrics: accuracy")

# Define callbacks for training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=model_config.early_stopping_patience,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        f'steelblast_{pipeline_config.model_choice.lower()}_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=model_config.reduce_lr_patience,
        min_lr=1e-7,
        verbose=1
    )
]

import time

# Train the model
print(f"Training {pipeline_config.model_choice} model...")
print(f"Epochs: {model_config.epochs}, Batch size: {preprocess_config.batch_size}")

train_start = time.time()
history = model.fit(
    train_dataset,
    epochs=model_config.epochs,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)
train_end = time.time()
training_time_seconds = float(train_end - train_start)
print(f"\nTraining completed in {training_time_seconds:.2f} seconds!")


## Evaluate Model Performance

This cell evaluates the trained model on the held-out test split, generates classification diagnostics, and saves both artifacts and metrics for reproducibility.

#### Approach
- Evaluate final loss and accuracy on `test_dataset`.
- Convert sigmoid outputs to binary predictions using a `0.5` decision threshold.
- Compute detailed metrics (`accuracy`, `precision`, `recall`, `f1`).
- Generate a full classification report and confusion matrix.
- Save the trained model to disk using a model name derived from `pipeline_config.model_choice`.
- Export run metadata and evaluation results to a timestamped JSON file.
- Print a final summary block for quick comparison across model runs.



In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
print(f"\nTest Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")

# Get predictions
# Use the model to predict on batches from the test dataset
predictions = model.predict(test_dataset, verbose=0).ravel()
y_pred = (predictions >= 0.5).astype(int)

# Extract true labels from the dataset
true_labels = np.concatenate([y for x, y in test_dataset], axis=0)

# Calculate detailed metrics
accuracy = accuracy_score(true_labels, y_pred)
precision = precision_score(true_labels, y_pred, average='weighted')
recall = recall_score(true_labels, y_pred, average='weighted')
f1 = f1_score(true_labels, y_pred, average='weighted')

print(f"\nDetailed Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Classification report
print(f"\nClassification Report:")
report_dict = classification_report(
    true_labels,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)
report_text = classification_report(
    true_labels,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)
print(report_text)

# Confusion Matrix
cm = confusion_matrix(true_labels, y_pred)
print(f"Confusion Matrix:\n{cm}")

# Save the trained model
model_path = f'steelblast_{pipeline_config.model_choice.lower()}.h5'
model.save(model_path)
print(f"Model saved to {model_path}")

# Save metrics to JSON
metrics = {
    "model": pipeline_config.model_choice,
    "image_size": preprocess_config.image_size,
    "classification_report": report_dict,
    "dataset_info": {
        "train_samples": len(train_paths),
        "val_samples": len(val_paths),
        "test_samples": len(test_paths),
        "classes": CLASS_NAMES
    },
    "confusion_matrix": cm.tolist(),
    "training_epochs": len(history.history['loss']),
    "training_time_seconds": training_time_seconds,
    "batch_size": preprocess_config.batch_size
}

from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
metrics_path = f"steelblast_{pipeline_config.model_choice.lower()}_metrics_{timestamp}.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"Metrics saved to {metrics_path}")


# Display final summary
print("\n" + "="*60)
print(f"FINAL SUMMARY - {pipeline_config.model_choice}")
print("="*60)
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Loss:      {test_loss:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")
print("="*60)


## Explainable AI (XAI): Occlusion Sensitivity Heatmaps

This section explains model decisions by masking small image regions and measuring how much the prediction confidence drops. Regions that cause larger drops are treated as more influential.

#### Approach
- Use test images and the trained model to compute baseline prediction confidence.
- Slide an occlusion window over the image grid.
- Re-run inference after each occlusion.
- Measure confidence drop for the predicted class.
- Aggregate drops into a heatmap and normalize it.
- Save visual overlays grouped by outcome type: `true_positive`, `true_negative`, `false_positive`, `false_negative`.

#### Parameters
- `max_samples`
  - Maximum number of test images to explain.
- `occlusion_size`
  - Size of the square occlusion patch in pixels.
- `stride`
  - Step size for moving the occlusion patch.
- `occlusion_value`
  - Pixel value used to mask occluded regions.
- `threshold`
  - Decision boundary for binary predictions (sigmoid output).


In [ ]:
# Occlusion Sensitivity XAI
from collections import defaultdict

# Configuration
max_samples = 16
occlusion_size = 32
stride = 16
occlusion_value = 0.0
threshold = 0.5

xai_output_dir = Path("steelblast_transfer_learning_occlusion_heatmaps")
outcome_dirs = {
    "true_positive": xai_output_dir / "true_positive",
    "true_negative": xai_output_dir / "true_negative",
    "false_positive": xai_output_dir / "false_positive",
    "false_negative": xai_output_dir / "false_negative",
}
for directory in outcome_dirs.values():
    directory.mkdir(parents=True, exist_ok=True)

def predict_sigmoid_score(image_rgb_float: np.ndarray) -> float:
    """Return sigmoid score for class 1 (not-good) for one image."""
    x = preprocess_fn(image_rgb_float.copy())
    x = np.expand_dims(x, axis=0)
    score = model.predict(x, verbose=0).ravel()[0]
    return float(score)

def confidence_for_class(score_class1: float, predicted_class: int) -> float:
    """Return confidence of the predicted class for binary sigmoid output."""
    return score_class1 if predicted_class == 1 else (1.0 - score_class1)

def outcome_bucket(y_true: int, y_hat: int) -> str:
    if y_true == 1 and y_hat == 1:
        return "true_positive"
    if y_true == 0 and y_hat == 0:
        return "true_negative"
    if y_true == 0 and y_hat == 1:
        return "false_positive"
    return "false_negative"

def occlusion_sensitivity_map(image_rgb_float: np.ndarray, predicted_class: int) -> np.ndarray:
    """Compute occlusion sensitivity map using confidence drop over grid patches."""
    h, w, _ = image_rgb_float.shape
    base_score = predict_sigmoid_score(image_rgb_float)
    base_conf = confidence_for_class(base_score, predicted_class)

    heat = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)

    for top in range(0, h, stride):
        for left in range(0, w, stride):
            bottom = min(top + occlusion_size, h)
            right = min(left + occlusion_size, w)

            occluded = image_rgb_float.copy()
            occluded[top:bottom, left:right, :] = occlusion_value

            occ_score = predict_sigmoid_score(occluded)
            occ_conf = confidence_for_class(occ_score, predicted_class)
            drop = max(0.0, base_conf - occ_conf)

            heat[top:bottom, left:right] += drop
            counts[top:bottom, left:right] += 1.0

    counts[counts == 0.0] = 1.0
    heat = heat / counts
    heat = heat - heat.min()
    max_val = heat.max()
    if max_val > 0:
        heat = heat / max_val
    return heat

num_samples = min(max_samples, len(test_paths))
indices = np.linspace(0, len(test_paths) - 1, num=num_samples, dtype=int)
saved_per_bucket = defaultdict(int)

print(f"Generating occlusion heatmaps for {num_samples} test images...")
print(f"Output directory: {xai_output_dir.resolve()}")

for i, idx in enumerate(indices, start=1):
    image_path = test_paths[idx]
    y_true = int(y_test[idx])

    # Load and resize image to model input size
    image = load_img(image_path, target_size=preprocess_config.image_size)
    image_rgb = img_to_array(image).astype(np.float32)

    score_class1 = predict_sigmoid_score(image_rgb)
    y_hat = int(score_class1 >= threshold)
    conf = confidence_for_class(score_class1, y_hat)

    heatmap = occlusion_sensitivity_map(image_rgb, y_hat)
    bucket = outcome_bucket(y_true, y_hat)
    saved_per_bucket[bucket] += 1

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image_rgb.astype(np.uint8))
    axes[0].set_title(f"Original\nTrue={CLASS_NAMES[y_true]}")
    axes[0].axis("off")

    hm = axes[1].imshow(heatmap, cmap="jet")
    axes[1].set_title("Occlusion Sensitivity")
    axes[1].axis("off")
    fig.colorbar(hm, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(image_rgb.astype(np.uint8))
    axes[2].imshow(heatmap, cmap="jet", alpha=0.45)
    axes[2].set_title(
        f"Overlay\nPred={CLASS_NAMES[y_hat]} | Conf={conf:.3f}"
    )
    axes[2].axis("off")

    fig.suptitle(
        f"{pipeline_config.model_choice} | Sample {i}/{num_samples} | {bucket}",
        fontsize=11
    )
    fig.tight_layout()

    out_name = f"{pipeline_config.model_choice.lower()}_{idx:04d}_{bucket}.png"
    out_path = outcome_dirs[bucket] / out_name
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

print("\nSaved heatmaps per outcome:")
for key in ["true_positive", "true_negative", "false_positive", "false_negative"]:
    print(f"  {key}: {saved_per_bucket[key]}")
print(f"\nDone. Heatmaps saved under: {xai_output_dir.resolve()}")

## 12. Model Comparison and Fine-tuning Tips

### Switching Between Models
To use a different pre-trained model, modify the `model_choice` parameter in the configuration cell:
- **VGG16**: Simple, interpretable, good baseline (224×224)
- **InceptionV3**: Efficient multi-scale feature extraction (299×299)
- **ResNet50**: Better performance with residual connections (224×224)

### Fine-tuning Strategies
1. **Unfreeze Later Layers**: After initial training, unfreeze the last few layers of the base model and fine-tune with a lower learning rate
2. **Data Augmentation**: Add rotation, zoom, and horizontal flip during training for better generalization
3. **Adjust Dropout Rates**: Increase dropout if overfitting occurs
4. **Learning Rate Tuning**: Use the ReduceLROnPlateau callback to adaptively reduce learning rate

### Expected Performance
- VGG16: ~85-90% accuracy (lower complexity)
- InceptionV3: ~88-92% accuracy (balanced)
- ResNet50: ~90-93% accuracy (best overall)